In [ ]:
%pip install xarray fsspec s3fs netCDF4

In [ ]:
import os
import xarray as xr
import fsspec
from pathlib import Path

# Brazil bounding box
LAT_MIN = -39.0208
LAT_MAX = 18.2292
LON_MIN = -94.1875
LON_MAX = 37.0625

# NASA POWER AWS S3 URL (Monthly Zarr)
URL = 'https://nasa-power.s3.us-west-2.amazonaws.com/syn1deg/temporal/power_syn1deg_monthly_temporal_lst.zarr'

# Output directory
OUTPUT_DIR = Path(r"Q:\My Drive\Brazil_NASA_POWER_Monthly")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Date range for 2025
TIME_START = "2025-01-01"
TIME_END = "2025-12-31"

print("=" * 80)
print("NASA POWER Download - Brazil 2025")
print("=" * 80)
print(f"Source: {URL}")
print(f"Region: Brazil ({LAT_MIN}°, {LON_MIN}°) to ({LAT_MAX}°, {LON_MAX}°)")
print(f"Time: {TIME_START} to {TIME_END}")
print(f"Output: {OUTPUT_DIR}")
print("=" * 80)

In [ ]:
# Open the Zarr dataset
print("\nOpening NASA POWER Zarr dataset...")
try:
    ds = xr.open_dataset(URL, engine='zarr')
    print("✓ Dataset opened successfully!")
    print(f"\nDataset dimensions: {dict(ds.dims)}")
    print(f"Total variables: {len(list(ds.data_vars))}")
    print(f"\nVariables: {list(ds.data_vars)}")
except Exception as e:
    print(f"✗ Error opening dataset: {e}")
    raise

In [ ]:
# Function to download one variable for 2025
def download_variable(ds, var_name):
    """
    Extract and save one variable for Brazil 2025.
    Matches Indonesia processing: one NetCDF per variable.
    """
    output_file = OUTPUT_DIR / f"{var_name}.nc"
    
    # Skip if already exists
    if output_file.exists():
        print(f"⏭️  Skipping {var_name} (already exists)")
        return True, "exists"
    
    print(f"\n📥 Processing {var_name}...")
    
    try:
        # Select Brazil region and 2025 time range
        ds_region = ds[var_name].sel(
            lat=slice(LAT_MIN, LAT_MAX),
            lon=slice(LON_MIN, LON_MAX),
            time=slice(TIME_START, TIME_END)
        ).load()
        
        print(f"   Region shape: {ds_region.shape}")
        print(f"   Time range: {ds_region.time.values[0]} to {ds_region.time.values[-1]}")
        print(f"   Months: {len(ds_region.time)}")
        
        # Save to NetCDF
        ds_region.to_netcdf(output_file)
        
        file_size = output_file.stat().st_size / 1024  # KB
        print(f"   ✓ Saved: {output_file.name} ({file_size:.1f} KB)")
        
        return True, None
        
    except Exception as e:
        print(f"   ✗ Error: {str(e)}")
        return False, str(e)

In [ ]:
# Download all 47 variables
print("\n" + "=" * 80)
print("DOWNLOADING ALL VARIABLES")
print("=" * 80)

variables = list(ds.data_vars)
success_count = 0
failed_count = 0
skipped_count = 0
failed_vars = []

for i, var in enumerate(variables, 1):
    print(f"\n[{i}/{len(variables)}] {var}")
    
    success, error = download_variable(ds, var)
    
    if success:
        if error == "exists":
            skipped_count += 1
        else:
            success_count += 1
    else:
        failed_count += 1
        failed_vars.append((var, error))

# Summary
print("\n" + "=" * 80)
print("DOWNLOAD COMPLETE")
print("=" * 80)
print(f"Total variables: {len(variables)}")
print(f"Successfully downloaded: {success_count}")
print(f"Already existed (skipped): {skipped_count}")
print(f"Failed: {failed_count}")

if failed_vars:
    print("\n❌ Failed variables:")
    for var, error in failed_vars:
        print(f"   • {var}: {error}")
else:
    print("\n✅ All variables downloaded successfully!")

print("\n" + "=" * 80)
print(f"Output directory: {OUTPUT_DIR}")
print(f"Files created: {success_count} NetCDF files (one per variable)")
print("\nNext steps:")
print("1. Process NetCDFs to monthly GeoTIFFs")
print("2. Resample to 0.02° (if needed)")
print("3. Upload to covariables2 database")
print("=" * 80)

In [ ]:
# Close dataset
ds.close()
print("Dataset closed.")